# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's enumerate the record sets, fields, columns, and their `@id` values.

In [ ]:
# Get all record sets in the dataset (as objects with .id)
record_sets = dataset.metadata.record_sets
print("Available record sets and their @id:")
for rs in record_sets:
    print(f"- RecordSet id: {rs.id} | Name: {rs.name}")

# For each record set, print available fields/columns
for rs in record_sets:
    print(f"\nFields in RecordSet {rs.name} ({rs.id}):")
    fields = rs.fields
    for fld in fields:
        print(f"  Field: {fld.name} | @id: {fld.id} | dataType: {fld.data_type}")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis. Refer to record set and field `@id` values from the overview.

We'll extract all tabular data as DataFrames.

In [ ]:
# Construct a list of record set IDs
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}
print("Loading records from each record set...")
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nDataFrame for RecordSet {record_set_id}:")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps—such as filtering records, normalizing numeric fields, and grouping by attributes—using column `@id` as reference.

Let's pick a numeric field from the first record set (e.g., age at diagnosis) and perform filtering and normalization.

In [ ]:
# Choose a record set and numeric field based on IDs found in overview
if record_set_ids:
    rs_id = record_set_ids[0]  # example: Table 1
    df = dataframes[rs_id]
    # Find numeric fields
    rs_obj = [rs for rs in dataset.metadata.record_sets if rs.id == rs_id][0]
    numeric_fields = [fld for fld in rs_obj.fields if fld.data_type in ['Integer', 'Float', 'Number']]
    if numeric_fields:
        numeric_field_id = numeric_fields[0].id
        print(f"Using numeric field @id: {numeric_field_id}")
        # Use column name that matches the @id (mlcroissant maps @id to DataFrame columns)
        threshold = 10
        if numeric_field_id in df.columns:
            filtered_df = df[df[numeric_field_id] > threshold]
            print(f"Filtered records with {numeric_field_id} > {threshold}:")
            print(filtered_df.head())
            # Normalization
            field_norm = f"{numeric_field_id}_normalized"
            filtered_df[field_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(f"Normalized {numeric_field_id} for filtered records:")
            print(filtered_df[[numeric_field_id, field_norm]].head())
            # Try to group by presence of e.g. 'infertility' (choose string field from fields)
            group_fields = [fld for fld in rs_obj.fields if fld.data_type == 'Boolean' or fld.data_type == 'Text']
            if group_fields:
                group_field_id = group_fields[0].id
                if group_field_id in filtered_df.columns:
                    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
                    print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
                    print(grouped_df.head())
                else:
                    print(f"Group field '@id: {group_field_id}' not present in DataFrame.")
    else:
        print("No numeric fields found for EDA.")
else:
    print('No record sets found.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Let's plot the distribution of the chosen numeric field and relationship with the group field.

In [ ]:
# Simple histogram and bar plot for EDA
if record_set_ids and numeric_fields:
    rs_id = record_set_ids[0]
    df = dataframes[rs_id]
    numeric_field_id = numeric_fields[0].id
    if numeric_field_id in df.columns:
        plt.figure(figsize=(6,4))
        df[numeric_field_id].dropna().hist(bins=5)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
        plt.show()
        # Bar plot by group field if possible
        group_fields = [fld for fld in rs_obj.fields if fld.data_type == 'Boolean' or fld.data_type == 'Text']
        if group_fields and group_fields[0].id in df.columns:
            group_field_id = group_fields[0].id
            group_means = df.groupby(group_field_id)[numeric_field_id].mean()
            group_means.plot(kind='bar')
            plt.title(f"Mean {numeric_field_id} by {group_field_id}")
            plt.xlabel(group_field_id)
            plt.ylabel(f"Mean {numeric_field_id}")
            plt.show()
    else:
        print(f"Numeric field '@id: {numeric_field_id}' not present in DataFrame.")
else:
    print('No visualization possible; missing numeric fields.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded the clinical dataset defined by the Croissant schema.
- Explored available record sets and their fields, referencing all via `@id`.
- Extracted tables and performed filtering, normalization, and grouping using record set and field `@id` values.
- Visualized numeric field distributions and group differences.

Due to the small, unique cohort (3 patients), this exploration is best suited for demonstration and descriptive analysis. Further research is encouraged for larger and more diverse cohorts.